# Downloading the Data

In [1]:
from datasets import load_dataset
import os

# Define the local directory to save the datasets for cluster upload
SAVE_DIR = "odqa_data"
os.makedirs(SAVE_DIR, exist_ok=True)

In [2]:
print("Downloading SQUAD-TR dataset...")

# Added trust_remote_code=True to allow the custom dataset script to run
squad_tr_open_qa = load_dataset("boun-tabi/squad_tr", "openqa")

# Display the dataset structure (train and validation splits)
print("SQUAD-TR Structure:")
print(squad_tr_open_qa)

# Save to disk
squad_tr_path = os.path.join(SAVE_DIR, "squad_tr")
squad_tr_open_qa.save_to_disk(squad_tr_path)
print(f"Saved SQUAD-TR to {squad_tr_path}")

/cta/users/buse/miniconda3/envs/odqa_env/lib/python3.10/site-packages/datasets/load.py:1429: FutureWarning: The repository for boun-tabi/squad_tr contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/boun-tabi/squad_tr
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


SQUAD-TR Structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/130319 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11873 [00:00<?, ? examples/s]

Saved SQUAD-TR to odqa_data/squad_tr


In [3]:
print("Downloading Turkish Wikipedia dump (20230901.tr)...")

# Load the specific 2023 subset from graelo/wikipedia
wiki_tr = load_dataset("graelo/wikipedia", "20230901.tr", split="train")

# Display the dataset structure (should show ~531k rows)
print("Wikipedia TR Structure:")
print(wiki_tr)

# Save to disk
wiki_tr_path = os.path.join(SAVE_DIR, "wiki_20230901_tr")
wiki_tr.save_to_disk(wiki_tr_path)
print(f"Saved Wikipedia TR to {wiki_tr_path}")

/cta/users/buse/miniconda3/envs/odqa_env/lib/python3.10/site-packages/datasets/load.py:1429: FutureWarning: The repository for graelo/wikipedia contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/graelo/wikipedia
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Wikipedia TR Structure:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 530830
})


Saving the dataset (0/2 shards):   0%|          | 0/530830 [00:00<?, ? examples/s]

Saved Wikipedia TR to odqa_data/wiki_20230901_tr


# Preprocessing the Dataset

In [4]:
import string
import os

def chunk_passages(batch, indices, source_name):
    titles = batch['title']
    texts = batch['text']
    source_ids = batch['id']
    
    chunked_titles = []
    chunked_texts = []
    chunked_ids = []
    
    # Paper specification: 75 words per chunk to prevent BERT truncation
    chunk_size = 75 
    
    for idx, (title, text, source_id) in enumerate(zip(titles, texts, source_ids)):
        source_index = indices[idx]

        # Skip if there is no text to chunk
        if not text:
            continue
            
        # Enhanced Whitespace Tokenizer (applied only to the text)
        # Pad punctuation with spaces so they are treated as distinct tokens
        processed_text = text
        for punc in string.punctuation:
            processed_text = processed_text.replace(punc, f" {punc} ")
            
        # Split purely by whitespace
        tokens = processed_text.split()
        
        # Split into equal chunks of 75 words
        for chunk_idx in range(0, len(tokens), chunk_size):
            chunk = tokens[chunk_idx:chunk_idx + chunk_size]
            
            # Keep the title separate for FiD
            chunked_titles.append(title)
            
            # Join the tokens back into a string chunk
            chunk_str = " ".join(chunk)
            
            # Clean up the punctuation spacing for readability 
            chunk_str = chunk_str.replace(" .", ".").replace(" ,", ",").replace(" ' ", "'")
            
            chunked_texts.append(chunk_str)
            chunked_ids.append(f"{source_name}-{source_id}-{source_index}-{chunk_idx // chunk_size}")
            
    return {"title": chunked_titles, "text": chunked_texts, "id": chunked_ids}


def chunk_wikipedia_passages(batch, indices):
    return chunk_passages(batch, indices, "wiki")

print("Chunking Turkish Wikipedia into 75-word segments...")

# Apply the mapping function to the dataset
wiki_tr_chunked = wiki_tr.map(
    chunk_wikipedia_passages, 
    batched=True, 
    with_indices=True,
    remove_columns=wiki_tr.column_names, 
    desc="Chunking Wikipedia passages"
)

print("New Chunked Wikipedia TR Structure:")
print(wiki_tr_chunked)

# Save the preprocessed chunks back to disk
chunked_path = os.path.join(SAVE_DIR, "wiki_20230901_tr_chunked")
wiki_tr_chunked.save_to_disk(chunked_path)
print(f"Saved chunked Wikipedia TR to {chunked_path}")


Chunking Turkish Wikipedia into 75-word segments...
New Chunked Wikipedia TR Structure:
Dataset({
    features: ['id', 'title', 'text'],
    num_rows: 2281587
})


Saving the dataset (0/3 shards):   0%|          | 0/2281587 [00:00<?, ? examples/s]

Saved chunked Wikipedia TR to odqa_data/wiki_20230901_tr_chunked


In [5]:
# Show a sample from the training set to verify the content

print("Sample from Chunked Wikipedia TR:")
print(wiki_tr_chunked[0])

Sample from Chunked Wikipedia TR:
{'id': 'wiki-10-0-0', 'title': 'Cengiz Han', 'text': "Cengiz Han ( doğum adıyla Temuçin, - 25 Ağustos 1227 ), tarihin bitişik sınırlara sahip en büyük kara imparatorluğu olan Moğol İmparatorluğu'nun kurucusu ve ilk büyük kağanı olan Moğol komutan ve hükümdardır. Hükümdarlığı döneminde gerçekleştirdiği hiçbir savaşı kaybetmeyen Cengiz Han, dünya tarihinin en başarılı askeri liderlerinden birisi olarak kabul edilmektedir. 13. yüzyılın başında Orta Asya'daki tüm göçebe bozkır kavimlerini birleştirip bir ulus hâline getirerek Moğol"}


In [7]:
import pandas as pd
from datasets import Dataset, concatenate_datasets

print("Extracting unique context passages from SQUAD-TR...")

# Convert the train split to a pandas DataFrame for easy deduplication
squad_train_df = squad_tr_open_qa['train'].to_pandas()

# Filter down to just titles and contexts, drop duplicates, and rename 'context' to 'text' 
# so it matches the input format expected by our chunk_wikipedia_passages function
unique_squad_df = squad_train_df[['id', 'title', 'context']].drop_duplicates().rename(columns={'context': 'text'})
#drop duplicates based on the 'text' column to ensure we only have unique passages, even if they have different titles or ids
unique_squad_df = unique_squad_df.drop_duplicates(subset=['text'])

# Convert back to a Hugging Face Dataset
squad_unique_ds = Dataset.from_pandas(unique_squad_df, preserve_index=False)
print(f"Extracted {len(squad_unique_ds)} unique passages from SQUAD-TR.")

print("Chunking the SQUAD-TR passages into 75-word segments...")
# Apply the exact same chunking function we used on the Wikipedia dump
squad_chunked = squad_unique_ds.map(
    lambda batch, indices: chunk_passages(batch, indices, "squad"),
    batched=True, 
    with_indices=True,
    remove_columns=squad_unique_ds.column_names,
    desc="Chunking SQUAD-TR passages"
)

print("Concatenating Wikipedia and SQUAD-TR chunks...")
# Combine both chunked datasets into the final knowledge source
final_knowledge_source = concatenate_datasets([wiki_tr_chunked, squad_chunked])

print("\nFinal Knowledge Source Structure:")
print(final_knowledge_source)

# Save the final, combined, and chunked dataset to disk for the cluster
final_path = os.path.join(SAVE_DIR, "final_knowledge_source_chunked")
final_knowledge_source.save_to_disk(final_path)
print(f"\nSaved final combined knowledge source to {final_path}")
print("Data preparation complete. Ready for offline indexing!")

Extracting unique context passages from SQUAD-TR...
Extracted 19028 unique passages from SQUAD-TR.
Chunking the SQUAD-TR passages into 75-word segments...


Chunking SQUAD-TR passages:   0%|          | 0/19028 [00:00<?, ? examples/s]

Concatenating Wikipedia and SQUAD-TR chunks...

Final Knowledge Source Structure:
Dataset({
    features: ['id', 'title', 'text'],
    num_rows: 2320816
})


Saving the dataset (0/3 shards):   0%|          | 0/2320816 [00:00<?, ? examples/s]


Saved final combined knowledge source to odqa_data/final_knowledge_source_chunked
Data preparation complete. Ready for offline indexing!


In [ ]:
# Show a sample from the training set to verify the content
squad_chunked
print("Sample from Chunked SQUAD-TR:")
print(squad_chunked[0])

Sample from Chunked SQUAD-TR:
{'id': 'squad-56be85543aeaaa14008c9063-0-0', 'title': 'Beyonce', 'text': "Beyoncé Giselle Knowles - Carter ( d. 4 Eylül 1981 ), ABD'li şarkıcı, söz yazarı, prodüktör ve aktris. Houston, Teksas'ta doğup büyüdü, çocukken çeşitli şarkı ve dans yarışmalarında sahne aldı ve 1990'ların sonlarında R & B kız grubu Destiny's Child'ın solisti olarak ün kazandı. Babası Mathew Knowles tarafından yönetilen grup tüm zamanların en çok satan kız gruplarından"}


In [14]:
squad_chunked.shape

(39229, 3)

In [1]:
# Count the lines from the jsonl file
for i in range(8):

    with open(f'fid_train_data_{i}_8.jsonl', 'r') as f:
        lines = f.readlines()
        print(f"Number of lines in the {i}th jsonl file: {len(lines)}")

Number of lines in the 0th jsonl file: 16290
Number of lines in the 1th jsonl file: 16290
Number of lines in the 2th jsonl file: 16290
Number of lines in the 3th jsonl file: 16290
Number of lines in the 4th jsonl file: 16290
Number of lines in the 5th jsonl file: 16290
Number of lines in the 6th jsonl file: 16290
Number of lines in the 7th jsonl file: 16289


In [3]:
# Merge the 8 jsonl files into one, 
# memory efficient version
import json

# Save the merged data into a new jsonl file
with open('fid_train_data_merged.jsonl', 'w') as f:
    for i in range(8):
        with open(f'fid_train_data_{i}_8.jsonl', 'r') as g:
            for line in g:
                f.write(json.dumps(json.loads(line)) + '\n')

In [6]:
lines[234]

'{"id": "5a8dbd49df8bba001a0f9bb9", "question": "T\\u00fcm se\\u00e7imlerde neler bulunur?", "answers": [], "real_ctx": {"title": "The_Legend_of_Zelda:_Twilight_Princess", "text": "Oyundan 20 m\\u00fczikal se\\u00e7imini i\\u00e7eren bir CD, Amerika Birle\\u015fik Devletleri\'nde GameStop \\u00f6n sipari\\u015f bonusu olarak mevcuttu; Japonya, Avrupa ve Avustralya\'daki t\\u00fcm paketlere dahil edildi. [al\\u0131nt\\u0131 gerekli]"}, "ctxs": [{"title": "Keith Mitchell", "text": "Mitchell liderli\\u011finde Parti, 1999 y\\u0131l\\u0131 Ocak ay\\u0131nda yap\\u0131lan erken se\\u00e7imlerde t\\u00fcm 15 sandalyeyi kazand\\u0131 ve NNP Kas\\u0131m 2003 se\\u00e7imlerinde iktidar\\u0131 \\u00fc\\u00e7\\u00fcnc\\u00fc d\\u00f6nemde kazand\\u0131. Yeni Ulusal Parti, Ulusal Demokratik Kongresi ( NDC ) taraf\\u0131ndan 8 Temmuz 2008 tarihinde yap\\u0131lan genel se\\u00e7imlerde yenildi. NDC 11\'e kar\\u015f\\u0131 sadece d\\u00f6rt sandalye kazand\\u0131. \\u015eubat 2013 genel se\\u00e7imle

In [29]:
# Get a specific sample from the SQUAD-TR dataset to verify the content
sample = squad_tr_open_qa['train'].filter(lambda example: example['id'] == "5a8dbd49df8bba001a0f9bb9")
print(sample[0])

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

{'id': '5a8dbd49df8bba001a0f9bb9', 'title': 'The_Legend_of_Zelda:_Twilight_Princess', 'context': "Oyundan 20 müzikal seçimini içeren bir CD, Amerika Birleşik Devletleri'nde GameStop ön sipariş bonusu olarak mevcuttu; Japonya, Avrupa ve Avustralya'daki tüm paketlere dahil edildi. [alıntı gerekli]", 'question': 'Tüm seçimlerde neler bulunur?', 'answers': {'text': []}}


In [30]:
import pandas as pd
from collections import defaultdict
from datasets import load_dataset

squad_tr_open_qa = load_dataset("boun-tabi/squad_tr", "openqa")

# A dictionary containing the ids of questions with the same title in squad dataset
question_ids = defaultdict(list)
for i in range(len(squad_tr_open_qa['train'])):
    if squad_tr_open_qa['train'][i]['answers']['text'] != []:
        title = squad_tr_open_qa['train'][i]['title']
        question_id = squad_tr_open_qa['train'][i]['id']
        question_ids[(title)].append(question_id)
    else: 
        print(f"Question with id {squad_tr_open_qa['train'][i]['id']} has no answers, skipping.")

print(len(question_ids.keys()))
# Put in a table the number of ids for each title
df = pd.DataFrame([(title, len(ids)) for title, ids in question_ids.items()], columns=['Title', 'Number of IDs'])
df.sort_values(by='Number of IDs', ascending=False)

/cta/users/buse/miniconda3/envs/odqa_env/lib/python3.10/site-packages/datasets/load.py:1429: FutureWarning: The repository for boun-tabi/squad_tr contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/boun-tabi/squad_tr
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Question with id 5a8d7bf7df8bba001a0f9ab1 has no answers, skipping.
Question with id 5a8d7bf7df8bba001a0f9ab2 has no answers, skipping.
Question with id 5a8d7bf7df8bba001a0f9ab3 has no answers, skipping.
Question with id 5a8d7bf7df8bba001a0f9ab4 has no answers, skipping.
Question with id 5a8d7bf7df8bba001a0f9ab5 has no answers, skipping.
Question with id 5a8d800edf8bba001a0f9abb has no answers, skipping.
Question with id 5a8d800edf8bba001a0f9abc has no answers, skipping.
Question with id 5a8d800edf8bba001a0f9abd has no answers, skipping.
Question with id 5a8d800edf8bba001a0f9abe has no answers, skipping.
Question with id 5a8d800edf8bba001a0f9abf has no answers, skipping.
Question with id 5a8d8412df8bba001a0f9ac5 has no answers, skipping.
Question with id 5a8d8412df8bba001a0f9ac6 has no answers, skipping.
Question with id 5a8d8412df8bba001a0f9ac7 has no answers, skipping.
Question with id 5a8d8412df8bba001a0f9ac8 has no answers, skipping.
Question with id 5a8d8412df8bba001a0f9ac9 has no

,Title,Number of IDs
7,New_York_City,817
12,American_Idol,790
0,Beyonce,753
1,Frédéric_Chopin,697
171,Queen_Victoria,677
...,...,...
35,Warsaw_Pact,52
112,Üzüm,48
133,Great_Plains,47
64,Tristan_da_Cunha,44


In [32]:
# Sample 5% of titles and get the corresponding question ids to create a validation set

validation_ids = [ids for title, ids in question_ids.items() if title in df.sample(frac=0.05, random_state=42)['Title'].tolist()] 
validation_ids = [item for sublist in validation_ids for item in sublist]
print(f"Number of validation ids: {len(validation_ids)}")
train_ids = [ids for title, ids in question_ids.items() if title not in df.sample(frac=0.05, random_state=42)['Title'].tolist()] 
train_ids = [item for sublist in train_ids for item in sublist]
print(f"Number of training ids: {len(train_ids)}")

Number of validation ids: 5090
Number of training ids: 81731


In [35]:
# creates a train and validation split from the merged jsonl file
# 5 percent for validation, 95 percent for training
# We will use the question_ids dictionary to ensure that all questions with the same title are in the same split
import json
with open(f'fid_train_data_merged.jsonl', 'r') as f:
    for line in f:
        line_dict = json.loads(line)
        if line_dict['id'] in validation_ids:
            with open('fid_train_data_validation.jsonl', 'a') as val_f:
                val_f.write(json.dumps(line_dict) + '\n')
        elif line_dict['id'] in train_ids:
            with open('fid_train_data_train.jsonl', 'a') as train_f:
                train_f.write(json.dumps(line_dict) + '\n')

In [36]:
# save validation and train ids to a json file for later use
with open('fid_validation_ids.json', 'w') as f:
    json.dump(validation_ids, f)
with open('fid_train_ids.json', 'w') as f:
    json.dump(train_ids, f)

In [23]:
import json
json.loads(lines[1453])

{'id': '56df8e3e38dc421700152040',
 'question': "Bell Washington'a hangi gün geldi?",
 'answers': ['Şubat 18'],
 'real_ctx': {'title': 'Alexander_Graham_Bell',
  'text': "Bu arada Elisha Gray de akustik telgrafla deneyler yapıyordu ve bir su vericisi kullanarak konuşmayı iletmenin bir yolunu buldu. 14 Şubat 1876'da Gray, su vericisi kullanan bir telefon tasarımı için ABD Patent Ofisi'ne ihbar etti. Aynı sabah, Bell'in avukatı Bell'in başvurusunu patent ofisine doldurdu. Kimin önce geldiği ve Gray'in daha sonra Bell'in patentinin önceliğine meydan okuduğu hakkında hatırı sayılır bir tartışma var. Bell 14 Şubat'ta Boston'daydı ve 26 Şubat'a kadar Washington'a gelmedi."},
 'ctxs': [{'title': 'Alexander_Graham_Bell',
   'text': "sayılır bir tartışma var. Bell 14 Şubat'ta Boston'daydı ve 26 Şubat'a kadar Washington'a gelmedi.",
   'id': 'squad-56df8e3e38dc42170015203f-2251-1',
   'score': 21.78374595516247},
  {'title': 'Telefon',
   'text': "aygıttır. İki bilim insanı, bu aygıtla ilk başar

In [ ]:
CUDA_VISIBLE_DEVICES=1,2 python external/fid/train_reader.py   --train_data fid_train_data_train.jsonl   --eval_data fid_train_data_validation.jsonl   --model_size base   --per_gpu_batch_size 1   --n_context 100   --text_maxlength 250   --answer_maxlength 20   --total_steps 80000   --warmup_steps 2000   --lr 5e-5   --optim adamw   --scheduler linear   --weight_decay 0.01   --use_checkpoint   --checkpoint_dir checkpoint   --name mt5_reader --eval_freq 4000 --save_freq 4000